# B5 — Sector analysis: knowledge intensity and informed retail flow

**Hypotheses stated before any sector result was seen.** A first set was
committed on 28 September 2026 (commit "Add sector analysis (B5)"). It was
revised to the set below, still before any result was seen; the git history
records both.

| | Hypothesis | Prediction |
|---|---|---|
| **H1 (primary)** | Retail option flow is more informative about earnings announcements in **knowledge-intensive industries** — those with a more technical workforce — because employees and others with industry expertise understand announcements better | In `CAR ~ flow + flow×KI + KI + flow×attention + …`, **flow×KI < 0**, net of attention. Variants: low-attention events, abnormal flow, short-dated OTM opening buys, excluding pharma & biotech. Tail test: extreme flow calls the direction more often in high-KI industries |
| **H2 (secondary)** | **Attention dilutes** informativeness: heavily covered firms attract uninformed traders whose volume masks any informed signal | **flow×attention > 0** (the negative flow effect weakens as attention rises) |
| **H3 (kept)** | Earnings are a **minor event for pharma & biotech**, whose largest news is regulatory | A smaller announcement move relative to normal volatility; a milder near-the-money IV crush |
| Exploratory | Flow × energy & industrials; tech's speculative demand | Reported as exploratory / descriptive |

**Knowledge intensity (KI)** is the industry's STEM employment share, from the
BLS OEWS national industry-specific estimates (step 1 below).

**What the data can and cannot show.** CBOE records that retail flow
predicted an announcement, not who traded or why. A positive H1 result is
evidence of *informed trading consistent with industry expertise*, not of
insider trading. Firms' own insiders are usually barred from trading before
earnings; the more plausible channel is expertise spilling across firms in
the same industry.

**Design.** Differences are tested directly, as interactions within one
regression. Size controls, year-quarter fixed effects, firm-and-date
clustered standard errors. KI and attention enter as z-scores. Pooled
2016–2025; eras are secondary.

**Reading the results.** Judge each hypothesis by whether its outcomes and
variants point the same way, not by one p-value. With this many rows, an
occasional p below 0.05 will appear by chance.

**Requires A1–A3**, plus the OEWS download in step 1.

## Setup

In [ ]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl

## Step 1 — Knowledge intensity from BLS OEWS

**One-time download.** From https://www.bls.gov/oes/tables.htm, under the
national industry-specific estimates, download the latest May release
(May 2024 or later; these use 2022 NAICS) and extract the whole zip into
`data/external/oews/`. The builder combines the national 4-digit, 3-digit
and sector files (`nat4d_*`, `nat3d_*`, `natsector_*`) and ignores the
ownership splits, the 5- and 6-digit file and `file_descriptions`. Reading
the 4-digit file (about 15 MB of Excel) takes a minute or so.

This builds each industry's STEM employment share (SOC groups 15, 17, 19
plus the three matching management occupations, over total employment,
summed over detailed occupations only) and prints the highest-STEM
industries. Sanity check: software, computer systems design, scientific
R&D and semiconductor manufacturing should be near the top.

In [ ]:
from pipeline.build_knowledge_intensity import build_stem_share, OUT_PATH

if not OUT_PATH.exists():
    build_stem_share()
else:
    stem = pl.read_parquet(OUT_PATH)
    print(stem.filter(pl.col("level") == 4).sort("stem_share", descending=True).head(10))

## Step 2 — Build the sector panel

One row per firm-event: the Test 1 panel plus sector, knowledge intensity
(matched at 4-digit NAICS where OEWS publishes it, else 3-digit, else sector;
the print-out shows how many events matched at each level), attention,
pre-event volatility, abnormal flow relative to the firm's own previous
events, and the option measures from the contract panel.

In [ ]:
from analysis.informed_trading import build_event_panel
from analysis.option_returns import PANEL_PATH
from analysis.sector_analysis import build_sector_panel

panel = build_event_panel(verbose=False)
contracts = pl.read_parquet(PANEL_PATH)
sp = build_sector_panel(panel, contracts)

## Step 3 — Check sectors and knowledge intensity

Confirm familiar names land in the right sector, and that median STEM share
ranks sectors sensibly (technology and pharma & biotech high, consumer &
retail low). If most events match only at sector level, KI is too coarse to
test H1 well — stop and check the NAICS codes.

In [ ]:
from analysis.sector_analysis import sector_check, sector_profile

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=120, float_precision=3, tbl_width_chars=220):
    print(sector_check(sp))
    print(sector_profile(sp))

## H1 and H2 — Knowledge intensity, attention, and informed flow

The primary row set is the first block (`primary: flow x KI, flow x
attention`): **H1** is `flow_z:ki_z` (predicted negative), **H2** is
`flow_z:attn_z` (predicted positive). The H1 variants follow, then
professional flow as a comparison. The SUE rows repeat everything with the
earnings surprise as the dependent variable.

The attention proxy measures attention with error, which biases its
interaction toward zero, so H2's estimate is conservative.

In [ ]:
from analysis.sector_analysis import test_knowledge_informativeness

ki = test_knowledge_informativeness(sp)
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=220, fmt_str_lengths=45):
    print(ki.select("hypothesis", "test", "outcome", "term", "n", "coef", "se", "p"))

### Tail test

Among the events with the most extreme abnormal retail flow (top 1% and 2%
by absolute value), how often does the flow call the direction of the
announcement return? Chance is 50%. H1 predicts a higher hit rate in
high-KI industries (top tercile of STEM share). This complements the
regressions; with few events in the tail, treat it as supporting evidence.

In [ ]:
from analysis.sector_analysis import tail_test

with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200):
    for t in (0.01, 0.02):
        print(tail_test(sp, tail=t).select("tail", "group", "events", "hits", "hit_rate", "p_vs_50pct"))

## H3 — Pharma & biotech: earnings as a minor event

**H3 predicts** negative coefficients on `event_ratio` and `abs_car`, and
**positive** coefficients on `atm_iv_change` (a milder crush is a less
negative change). Caution: the dispersion control behaves oddly for
loss-making firms, which biotech often are.

In [ ]:
from analysis.sector_analysis import test_biotech_minor_event

with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200):
    print(test_biotech_minor_event(sp).select("outcome", "term", "n", "coef", "se", "p"))

## Exploratory and descriptive

Not hypothesis tests. The energy & industrials flow interaction (the
original H3) and tech's speculative demand (the original H1), kept for
completeness and labelled accordingly in the results table.

In [ ]:
from analysis.sector_analysis import test_flow_interactions, test_speculative_demand

with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=220):
    print(test_flow_interactions(sp).select("test", "outcome", "term", "n", "coef", "p"))
    print(test_speculative_demand(sp).select("test", "outcome", "term", "n", "coef", "p"))

## Secondary: by era

The primary H1/H2 regression within each era, as a consistency check.
Samples are smaller, so read for direction rather than significance.

In [ ]:
from analysis.informed_trading import ERAS

by_era = pl.concat([test_knowledge_informativeness(sp.filter(pl.col("era") == e)).with_columns(pl.lit(e).alias("era"))
                    for e in ERAS], how="diagonal_relaxed")
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=220, fmt_str_lengths=45):
    print(by_era.filter(pl.col("test").str.starts_with("primary")).select("era", "outcome", "term", "n", "coef", "p"))

## Write results

`table12_sector_profile.csv`, `table12b_sector_membership.csv`,
`table13_sector_hypotheses.csv` (every test, labelled by hypothesis) and
`table13b_tail_test.csv` to `results/`.

In [ ]:
from analysis.sector_analysis import write_sector_results

sector_results = write_sector_results(sp)